<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/sparse_gp_regression_1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Sparse GP regression (1-D) with `gpc`

Inspired by GPy's [`sparse_GP_regression_1D`](https://gpy.readthedocs.io/en/latest/GPy.examples.html#GPy.examples.regression.sparse_GP_regression_1D) example.

Synthetic \(\sin\) data, DTCVAR sparse GP (default), free inducing inputs on a grid. GPy often picks inducing points for you via `num_inducing=`; here we pass `Z` explicitly.

```bash
pip install matplotlib ipywidgets   # or: pip install 'gpc[examples]'
```


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a6"


In [ ]:
%pip install matplotlib ipywidgets


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

rng = np.random.default_rng(0)
n, m_ind = 200, 8
X = rng.uniform(-3.0, 3.0, size=(n, 1))
X = np.sort(X, axis=0)
Y = np.sin(X) + 0.05 * rng.normal(size=X.shape)
Z = np.linspace(-2.5, 2.5, m_ind)[:, None]

k = gpc.kern.RBF(1)
ms = gpc.models.SparseGPRegression(X, Y, Z, kernel=k)  # approx='dtcvar'
ms.set_by_path("rbf.lengthscale", 1.0)
print("approx:", ms.approx, "paths:", ms.param_paths())
print("objective before:", ms.objective())
ms.optimize(iters=80, messages=True)
print("objective after:", ms.objective())
print(ms)


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

Xtest = np.linspace(-3.5, 3.5, 300)[:, None]
ax, _, _ = plot_1d(ms, X, Y, Xtest=Xtest, label="dtcvar mean")
Zplot = ms.inducing_inputs()
ax.plot(Zplot[:, 0], np.full(Zplot.shape[0], Y.min() - 0.2), "r|", markersize=14, label="inducing")
ax.legend(loc="best")
ax.set_title("Sparse GP regression 1-D (DTCVAR)")
plt.show()

mu_y, var_y = ms.predict(Xtest)
mu_f, var_f = ms.predict_noiseless(Xtest)
print("mean(var_y - var_f):", float(np.mean(var_y - var_f)),
      "likelihood.variance:", ms.get_by_path("likelihood.variance"))


## Optional: FITC on the same data

GPy's sparse example uses variational DTC by default; `gpc` also exposes `approx="fitc"` / `"dtc"`.


In [ ]:
mf = gpc.models.SparseGPRegression(X, Y, Z, kernel=gpc.kern.RBF(1), approx="fitc")
mf.set_by_path("rbf.lengthscale", 1.0)
mf.optimize(iters=80, messages=True)
print("FITC objective:", mf.objective())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(ms, X, Y, Xtest=Xtest, ax=axes[0], label="dtcvar")
axes[0].set_title("DTCVAR")
plot_1d(mf, X, Y, Xtest=Xtest, ax=axes[1], label="fitc")
axes[1].set_title("FITC")
plt.tight_layout()
plt.show()
